# SR-JEPA on Colab: V-JEPA 2-style super-resolution for CMS jets and CaloChallenge showers

This notebook runs the full [`super-resolution-JEPA`](https://github.com/rajveer43/super-resolution-JEPA) pipeline and saves every result to Google Drive:

`download → prepare → pretrain → train_sr → train_decoder → train_probe → train_e2e → evaluate`

| Stage | What it does |
|---|---|
| `download` | Fetches the raw data once to Drive: CERNBox for CMS jets, Zenodo for CaloChallenge. Complete files are skipped. Files already downloaded by the DiffLense notebook are reused. |
| `prepare` | Builds the HR truth, the noisy measured LR (sum-pool, noise, topo-clustering) and the train/val/test split, on local disk. |
| `pretrain` | **Stage 1**, V-JEPA 2 mask denoising in representation space (eq. 1). Self-supervised: no labels, no LR→HR pairs. |
| `train_sr` | **Stage 2**, the latent SR world model on the frozen encoder (V-JEPA 2-AC analogue), with teacher-forcing + rollout losses (eqs. 2–4). |
| `train_decoder` | **Stage 3**, a pixel decoder from latent tokens (+ the LR measurement) to HR energies. |
| `train_probe` | An attentive probe on frozen latents: quark/gluon for jets, E_inc for showers. |
| `train_e2e` | Ablation: the same network fine-tuned end to end (`e2e_jepa`) and trained from scratch (`e2e_scratch`). |
| `evaluate` | Energy closure, hit F1, physics observables, latent error and probe metrics against the truth, for every variant. |

**Data on the GPU.** Each split is loaded once into GPU memory, or RAM if it is too large, and batched by indexing; the log says where each split went.

**Every stage resumes.** After a disconnect, run the notebook from the top. Training continues from `last.pt` on Drive, and a finished stage only re-runs its final validation. The local cache is rebuilt the same way every time, so the split never changes.

**Before you start:** pick *Runtime → Change runtime type → GPU*. An A100 or L4 is best (bf16). A T4 works in fp32; halve the batch sizes if you run out of memory.

## 1. Settings
Every option lives in this one cell. Keep `SMOKE = True` for the first pass: a few minutes on a small subset, written to its own `smoke` folder, to check that everything runs. Then set `SMOKE = False` for the real run.

In [ ]:
DATASET   = "qg"               # "qg" = CMS quark/gluon jet images | "calo" = CaloChallenge DS2 electron showers
RUN_TAG   = "vjepa2-baseline"  # short description; becomes part of the experiment folder name
SMOKE     = True               # True: quick end-to-end check on a small subset (separate folder)
RUN_DATE  = None               # None = today (YYYY-MM-DD). Set it to resume an experiment started on another day.

# --- real-run settings (ignored when SMOKE = True) ---
MAX_EVENTS       = None        # None = all (qg: 139k jets; calo: 100k showers from dataset_2_1)
PRETRAIN_EPOCHS  = 100         # stage 1 (self-supervised); the longest stage
SR_EPOCHS        = 50          # stage 2
DECODER_EPOCHS   = 40          # stage 3
PROBE_EPOCHS     = 20
E2E_EPOCHS       = 40
RUN_E2E          = True        # the jepa-vs-scratch ablation (two more trainings)
PATIENCE         = 5           # early stopping (epochs without val improvement) for SR, decoder, probe, e2e
BATCH            = None        # None = by GPU: 256 on A100/H100, 128 on L4, 64 otherwise.
                               # Learning rates scale with sqrt(batch) automatically.
ENCODER          = {"dim": 384, "depth": 12, "heads": 6}   # ViT-S. ViT-B: dim 768, heads 12

DRIVE_ROOT = "/content/drive/MyDrive/GSoC_SR_JEPA"
# raw files shared with the DiffLense notebook (same datasets): reused if they are already there
DIFFLENSE_DATASETS = "/content/drive/MyDrive/GSoC_DiffLense/datasets"
REPO_URL   = "github.com/rajveer43/super-resolution-JEPA.git"
BRANCH     = "main"

## 2. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "no NVIDIA GPU visible")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to GPU: Runtime -> Change runtime type"
print("bf16:", torch.cuda.is_bf16_supported(including_emulation=False), "(False on a T4: training runs in fp32)")
gpu = torch.cuda.get_device_name(0)
if BATCH is None:
    BATCH = 256 if any(k in gpu for k in ("A100", "H100")) else 128 if "L4" in gpu else 64
print("batch size:", BATCH)

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Get the code
Clones the repo on the first run and pulls the latest commit after that. The repo is public, so no token is needed. If you make it private, add a Colab secret `GITHUB_TOKEN`; it is read from there and never printed.

In [ ]:
import os, subprocess
REPO = "/content/super-resolution-JEPA"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token or "\0", "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if not os.path.exists(REPO):
    git("clone", "--branch", BRANCH, url, REPO)
else:
    git("pull", url, BRANCH, cwd=REPO)
COMMIT = git("rev-parse", "--short", "HEAD", cwd=REPO).strip()
print("code at commit", COMMIT)
print(git("log", "-1", "--format=%s (%cd)", cwd=REPO))

## 5. Install dependencies
Colab already has PyTorch, so only the remaining packages are installed.

In [ ]:
!grep -v -E "^torch" {REPO}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

## 6. Experiment folders and the run helper
```
MyDrive/GSoC_SR_JEPA/
  datasets/<dataset>/raw/                    raw files (unless reused from GSoC_DiffLense/datasets)
  experiments/<dataset>/<date>_<run-tag>/    one experiment
      run_info.json
      logs/<stage>.log                       full console output of each stage
      pretrain/ sr/ decoder/ probe/          best.pt, last.pt, history.csv, config.json
      e2e_jepa/ e2e_scratch/
      evaluation/                            metrics.json, summary.md, examples.png, observables.png
  exports/<dataset>_<date>_<run-tag>_results.zip
```
The preprocessed cache is too large for Drive, so it stays on local disk (`/content/data/cache`) and is rebuilt identically after a disconnect.

In [ ]:
import datetime, json, os, sys
from IPython.display import Markdown, Image, display

date = RUN_DATE or datetime.date.today().isoformat()
tag = "smoke" if SMOKE else RUN_TAG
RUN_NAME = f"{date}_{tag}"
CONFIG = f"configs/{DATASET}.yaml"
EXP = f"{DRIVE_ROOT}/experiments/{DATASET}/{RUN_NAME}"
shared = f"{DIFFLENSE_DATASETS}/{DATASET}/raw"
RAW_DRIVE = shared if os.path.isdir(shared) and os.listdir(shared) else f"{DRIVE_ROOT}/datasets/{DATASET}/raw"
RAW_LOCAL = f"/content/data/raw/{DATASET}"
CACHE = f"/content/data/cache/{DATASET}_" + ("smoke" if SMOKE else f"{MAX_EVENTS or 'all'}")  # one cache per event count
for d in (EXP, f"{EXP}/logs", RAW_DRIVE, RAW_LOCAL, f"{DRIVE_ROOT}/exports"):
    os.makedirs(d, exist_ok=True)
print("raw data on Drive:", RAW_DRIVE)

SETS = [f"run_name={RUN_NAME}", f"paths.out={EXP}", f"paths.cache={CACHE}", f"paths.raw={RAW_LOCAL}",
        "num_workers=2", "device=cuda"]
if DATASET == "qg":  # jet energies fit fp16: the full train split stays on the GPU (falls back to fp32 if not)
    SETS += ["data.preload_dtype=float16"]
if SMOKE:
    SETS += ["data.max_events=2000", "pretrain.epochs=2", "sr.epochs=2", "decoder.epochs=2", "probe.epochs=2",
             "e2e.epochs=1", "eval.max_events=300"]
    if DATASET == "calo":
        SETS += ["data.test_max=500"]
else:
    SETS += [f"data.max_events={'null' if MAX_EVENTS is None else MAX_EVENTS}"]
    SETS += [f"encoder.{k}={v}" for k, v in ENCODER.items()]
    SETS += [
             f"pretrain.epochs={PRETRAIN_EPOCHS}", f"sr.epochs={SR_EPOCHS}", f"decoder.epochs={DECODER_EPOCHS}",
             f"probe.epochs={PROBE_EPOCHS}", f"e2e.epochs={E2E_EPOCHS}"]
    SETS += [f"{s}.patience={PATIENCE}" for s in ("sr", "decoder", "probe", "e2e")]
    SETS += [f"{s}.batch_size={BATCH}" for s in ("pretrain", "sr", "decoder", "e2e")]

def run(*stages, extra=(), log=None):
    """Run pipeline stages; stream the output here and append it to logs/<stage>.log on Drive."""
    cmd = [sys.executable, "-u", "-m", "sr_jepa", *stages, "-c", CONFIG, "--set", *SETS, *extra]
    path = f"{EXP}/logs/{log or '_'.join(stages)}.log"
    with open(path, "a") as f:
        f.write(f"\n===== {datetime.datetime.now():%Y-%m-%d %H:%M:%S} | commit {COMMIT} | {' '.join(stages)}\n")
        p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            f.write(line)
        p.wait()
    if p.returncode:
        raise RuntimeError(f"stage(s) {stages} failed (exit {p.returncode}); full log: {path}")

def history(stage, n=10):
    import pandas as pd
    path = f"{EXP}/{stage}/history.csv"
    if os.path.exists(path):
        display(pd.read_csv(path).tail(n))

# run_info.json: one entry per session (when, which code, which GPU, which settings)
info_path = f"{EXP}/run_info.json"
info = json.load(open(info_path)) if os.path.exists(info_path) else {
    "experiment": RUN_NAME, "dataset": DATASET, "started": datetime.datetime.now().isoformat(timespec="seconds"),
    "sessions": []}
info["sessions"].append({"time": datetime.datetime.now().isoformat(timespec="seconds"), "commit": COMMIT,
                         "gpu": torch.cuda.get_device_name(0), "smoke": SMOKE, "overrides": SETS})
json.dump(info, open(info_path, "w"), indent=2)
print("experiment folder:", EXP)

## 7. Download the dataset (once, to Drive)
- **qg:** the public CERNBox share `EYgmOkI9BjwxNqy`, which holds 3 parquet files of ML4Sci CMS jet images (139k jets, 3.7 GB).
- **calo:** Zenodo record 6366271, `dataset_2_1.hdf5` and `dataset_2_2.hdf5`, md5-checked.

This takes 5–15 minutes the first time; complete files are skipped later. The files are then copied to local disk, which reads much faster than Drive.

In [ ]:
run("download", extra=[f"paths.raw={RAW_DRIVE}"])
!rsync -a --info=progress2 "{RAW_DRIVE}/" "{RAW_LOCAL}/"
!ls -lh "{RAW_LOCAL}"

## 8. Prepare the cache
Builds the HR truth and the noisy measured LR: sum-pooled 4×4 for jets or 4×3 for showers, then noise and 4-2-0 topo-clustering. Splits are fixed by the seed, and scales come from the train split only. Skipped if the cache already exists.

In [ ]:
run("prepare")

## 9. Stage 1: self-supervised JEPA pretraining
The encoder sees each event with blocks of tokens dropped, and the predictor regresses the EMA target encoder's features of the dropped tokens (L1, eq. 1). Inputs mix the LR measurement, the intermediate grid and the truth. For jets, training runs at 64×64 until the cooldown and at 128×128 after it (progressive resolution).

**Watch `val_feat_std`:** it must stay well above 0. If it falls towards 0, the representation has collapsed. `val_loss` is measured at full resolution, so it is only comparable with itself from the cooldown on (the last 20 % of epochs, `train_pool=1`). `events_per_s` is the throughput: on an A100 it should be in the thousands.

In [ ]:
run("pretrain")
history("pretrain")

## 10. Stage 2: latent SR world model
The encoder is frozen. A block-causal predictor steps through the resolution ladder in latent space, from LR to the intermediate grid to HR. It is trained with teacher-forcing + rollout losses.

**Watch:** `val_loss` (the rollout from the LR alone) must fall clearly below `val_identity` (the latent error of not super-resolving). Training stops after `PATIENCE` epochs without improvement, and `best.pt` (the lowest `val_loss`) is what the next stages use.

In [ ]:
run("train_sr")
history("sr")

## 11. Stage 3: pixel decoder
Decodes latent tokens, plus the LR measurement, into HR energies. `val_autoencode` (decoding E(HR)) is the ceiling that `val_loss` (decoding the SR prediction) can reach.

In [ ]:
run("train_decoder")
history("decoder")

## 12. Attentive probe on frozen latents (quark/gluon for jets, E_inc for showers)

In [ ]:
run("train_probe")
history("probe")

## 13. Ablation: end-to-end fine-tuning and training from scratch
`e2e_jepa` starts from stages 1–3 and fine-tunes everything on pixels. `e2e_scratch` trains the same architecture from random weights. Comparing them with the frozen `jepa` variant shows what self-supervised pretraining contributes.

In [ ]:
if RUN_E2E:
    run("train_e2e")
    history("e2e_jepa", 5)
    history("e2e_scratch", 5)

## 14. Evaluate every variant against the HR truth

In [ ]:
run("evaluate")
display(Markdown(open(f"{EXP}/evaluation/summary.md").read()))

## 15. Figures

In [ ]:
for f in ("examples.png", "observables.png"):
    p = f"{EXP}/evaluation/{f}"
    if os.path.exists(p):
        print(f)
        display(Image(p, width=1000))

## 16. Export a results bundle
Zips the experiment into `exports/` on Drive: configs, logs, histories, metrics, figures and every `best.pt`. Per-epoch `last.pt` files are left out, except the pretrained encoder's, which is the model the later stages use.

In [ ]:
import zipfile
zip_path = f"{DRIVE_ROOT}/exports/{DATASET}_{RUN_NAME}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(EXP):
        for f in files:
            p = os.path.join(root, f)
            rel = os.path.relpath(p, EXP)
            if f.endswith(".tmp") or (f == "last.pt" and not rel.startswith("pretrain")):
                continue
            z.write(p, os.path.join(f"{DATASET}_{RUN_NAME}", rel))
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")